In [92]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [93]:
#load the dataset and clean basic columns

df = pd.read_csv("train.En.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df["tweet"] = df["tweet"].astype(str).fillna("")
texts = df["tweet"].str.lower()
labels = df["sarcastic"]

In [94]:
#split the data into training and validation sets (stratified)

X_train_text, X_val_text, y_train, y_val = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [95]:
#initialize the TF-IDF vectorizer with chosen parameters

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
    sublinear_tf=True
)

In [96]:
#fit TF-IDF on training data and transform both train and validation sets

X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)


Train shape: (2774, 7565)
Val shape: (694, 7565)


In [97]:
#check the number of features and sample vocabulary terms

feature_names = vectorizer.get_feature_names_out()
print("num features:", len(feature_names))
print("first 20 features:", feature_names[:20])

num features: 7565
first 20 features: ['00' '000' '10' '10 10' '10 mins' '10 minutes' '10 years' '100' '1000'
 '10pm' '10pm coincidentally' '10th' '11' '11 at' '12' '12 hours' '12 hr'
 '13' '13 year' '14']


In [98]:
#sample tweet and its corresponding TF-IDF representation

print("Tweet:", X_train_text.iloc[0])
print("Label:", y_train.iloc[0])
print("TF-IDF row:", X_train[0])

Tweet: @igreen95 thx for the play by play
Label: 1
TF-IDF row:   (0, 6456)	0.5392309318312463
  (0, 2190)	0.18185215580127118
  (0, 6038)	0.12723830800107794
  (0, 4835)	0.6917826303949002
  (0, 1102)	0.3065411783675111
  (0, 2241)	0.2957017193268068


In [99]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [100]:
# Train logistic regression with balanced class weights

model = LogisticRegression(max_iter=2000, class_weight="balanced")
model.fit(X_train, y_train)



LogisticRegression(class_weight='balanced', max_iter=2000)

In [101]:
# Generate predictions on the validation set
pred = model.predict(X_val)

In [102]:
# Evaluate model performance using the default threshold (0.5)
acc = accuracy_score(y_val, pred)
prec = precision_score(y_val, pred, pos_label=1, zero_division=0)
rec = recall_score(y_val, pred, pos_label=1, zero_division=0)
f1 = f1_score(y_val, pred, pos_label=1)

print("Accuracy:", acc)
print("Precision (sarcastic=1):", prec)
print("Recall (sarcastic=1):", rec)
print("F1 (sarcastic=1):", f1)


Accuracy: 0.6570605187319885
Precision (sarcastic=1): 0.3316062176165803
Recall (sarcastic=1): 0.3699421965317919
F1 (sarcastic=1): 0.34972677595628415


In [103]:
cm = confusion_matrix(y_val, pred)
print(cm)


[[392 129]
 [109  64]]


In [104]:
# Tune regularization strength (C) and select the best F1 score
best_C = None
best_f1 = -1
best_pred = None

for C in [0.1, 0.5, 1, 3, 10]:
    temp_model = LogisticRegression(
        max_iter=2000,
        C=C,
        class_weight="balanced",
        solver="liblinear"
    )
    temp_model.fit(X_train, y_train)
    temp_pred = temp_model.predict(X_val)

    temp_acc = accuracy_score(y_val, temp_pred)
    temp_prec = precision_score(y_val, temp_pred, pos_label=1, zero_division=0)
    temp_rec = recall_score(y_val, temp_pred, pos_label=1, zero_division=0)
    temp_f1 = f1_score(y_val, temp_pred, pos_label=1)

    print(f"C={C} | Acc={temp_acc:.3f} | Prec={temp_prec:.3f} | Rec={temp_rec:.3f} | F1={temp_f1:.3f}")

    if temp_f1 > best_f1:
        best_f1 = temp_f1
        best_C = C
        best_pred = temp_pred

print("\nBest C:", best_C, "Best F1:", best_f1)
print("Confusion matrix for best C:")
print(confusion_matrix(y_val, best_pred))


C=0.1 | Acc=0.627 | Prec=0.316 | Rec=0.428 | F1=0.364
C=0.5 | Acc=0.651 | Prec=0.330 | Rec=0.387 | F1=0.356
C=1 | Acc=0.656 | Prec=0.328 | Rec=0.364 | F1=0.345
C=3 | Acc=0.661 | Prec=0.324 | Rec=0.329 | F1=0.327
C=10 | Acc=0.664 | Prec=0.312 | Rec=0.289 | F1=0.300

Best C: 0.1 Best F1: 0.36363636363636365
Confusion matrix for best C:
[[361 160]
 [ 99  74]]


In [105]:
# Retrain logistic regression using the best C value
best_model = LogisticRegression(
    max_iter=2000,
    C=best_C,
    class_weight="balanced",
    solver="liblinear"
)
best_model.fit(X_train, y_train)

pred = best_model.predict(X_val)
print("Final check F1:", f1_score(y_val, pred, pos_label=1))


Final check F1: 0.36363636363636365


In [106]:
# Tune decision threshold to maximize F1 on the validation set
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

probs = best_model.predict_proba(X_val)[:, 1]

best_t = None
best_f1 = -1

for t in np.arange(0.05, 0.96, 0.05):
    preds = (probs >= t).astype(int)
    f1 = f1_score(y_val, preds, pos_label=1)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t
    print(f"t={t:.2f} | F1={f1:.3f}")

print("\nBest threshold:", best_t, "Best F1:", best_f1)



t=0.05 | F1=0.399
t=0.10 | F1=0.399
t=0.15 | F1=0.399
t=0.20 | F1=0.399
t=0.25 | F1=0.399
t=0.30 | F1=0.400
t=0.35 | F1=0.400
t=0.40 | F1=0.400
t=0.45 | F1=0.409
t=0.50 | F1=0.364
t=0.55 | F1=0.055
t=0.60 | F1=0.000
t=0.65 | F1=0.000
t=0.70 | F1=0.000
t=0.75 | F1=0.000
t=0.80 | F1=0.000
t=0.85 | F1=0.000
t=0.90 | F1=0.000
t=0.95 | F1=0.000

Best threshold: 0.45 Best F1: 0.40865384615384615


In [107]:
preds = (probs >= best_t).astype(int)

acc = accuracy_score(y_val, preds)
prec = precision_score(y_val, preds, pos_label=1, zero_division=0)
rec = recall_score(y_val, preds, pos_label=1, zero_division=0)
f1 = f1_score(y_val, preds, pos_label=1)

print("Threshold:", best_t)
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)
print(confusion_matrix(y_val, preds))



Threshold: 0.45
Accuracy: 0.2910662824207493
Precision: 0.25796661608497723
Recall: 0.9826589595375722
F1: 0.40865384615384615
[[ 32 489]
 [  3 170]]
